<a href="https://colab.research.google.com/github/takeuchi6459/Physical-AI/blob/main/03_run_plans_%E5%87%BA%E5%8A%9B%E5%89%8A%E9%99%A4%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03b プランJSONを歩かせて動画を表示する
Driveの `params.pkl`（学習済み）と `plans/*.json` を読み込み、全部歩かせて動画を画面に表示します（ファイル保存はしません）。GPUランタイムで、上から順に実行。

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
BASE = "/content/drive/MyDrive/physical-ai-final"
PARAMS_PATH = BASE + "/ckpt_g1/params.pkl"
PLAN_DIR = BASE + "/plans"
ENV_NAME = "G1JoystickFlatTerrain"

from google.colab import drive
drive.mount('/content/drive')
print("params:", os.path.exists(PARAMS_PATH))
print("plans:", sorted(os.listdir(PLAN_DIR)) if os.path.exists(PLAN_DIR) else "なし")

Mounted at /content/drive
params: True
plans: ['.ipynb_checkpoints', 'demo1_L.json', 'demo2_U.json', 'demo3_square.json', 'demo4_zigzag.json', 'demo5_short.json']


In [ ]:
!pip install -q mujoco mujoco_mjx brax playground mediapy
!pip install -q -U flax

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.6/30.6 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.9/356.9 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.4/750.4 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.

In [ ]:
import math, json, pickle, glob
import numpy as np, jax, jax.numpy as jp
import mujoco, mediapy as media
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics
from mujoco_playground import registry
from mujoco_playground.config import locomotion_params

print(jax.devices())
env_cfg = registry.get_default_config(ENV_NAME)
env = registry.load(ENV_NAME, config=env_cfg)

ppo_params = locomotion_params.brax_ppo_config(ENV_NAME)
nf = dict(ppo_params.network_factory) if "network_factory" in ppo_params else {}
state0 = jax.jit(env.reset)(jax.random.PRNGKey(0))
obs_shape = jax.tree_util.tree_map(lambda x: x.shape, state0.obs)
network = ppo_networks.make_ppo_networks(
    obs_shape, env.action_size,
    preprocess_observations_fn=running_statistics.normalize, **nf)
make_inference_fn = ppo_networks.make_inference_fn(network)

params = pickle.load(open(PARAMS_PATH, "rb"))
jit_inf   = jax.jit(make_inference_fn(params, deterministic=True))
jit_reset = jax.jit(env.reset)
jit_step  = jax.jit(env.step)
print("policy loaded")

[CudaDevice(id=0)]
mujoco_menagerie not found. Downloading...


Cloning mujoco_menagerie: ██████████| 100/100 [01:00<00:00]


Checking out commit 1b86ece576591213e2b666ebf59508454200ca97
Successfully downloaded mujoco_menagerie
Warp 1.18.0 initialized:
   CUDA Toolkit 13.4, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.18.0
Module mujoco.mjx.warp.ffi 4443472 load on device 'cuda:0' took 803.87 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 43a6a77 load on device 'cuda:0' took 9435.71 ms  (compiled)
Module _nxn_broadphase__locals__kernel_95773766 9577376 load on device 'cuda:0' took 650.69 ms  (compiled)
Module ccd_kernel_builder__locals__ccd_kernel_1a0e5f50 1a0e5f5 load on device 'cuda:0' took 34244.94 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_dfed0db2 dfed0db load on device 'cuda:0' took 2135.13 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 8d4d513 load on device 'cuda:0' took 128.23 ms  (compiled)
Module _friction_dof_

In [ ]:
def yaw_of(s):
    w, x, y, z = np.array(s.data.qpos)[3:7]
    return math.atan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))

def pos_of(s):
    return np.array(s.data.qpos)[:2]

def wrap(a):
    return (a + math.pi) % (2*math.pi) - math.pi

def snap(s):   # GPU上のstateは保持せず、描画に必要な最小限だけCPUへ
    return np.array(s.data.qpos), np.array(s.data.qvel)

def run_plan(plan_steps, vx=0.8, kp=2.0, seed=0, brake_sec=0.0, slow_dist=1.5):
    rng = jax.random.PRNGKey(seed)
    state = jit_reset(rng)
    traj = [snap(state)]

    # 目標は最初に「理想の経路」として固定する（実際の位置のずれに引きずられない）
    p, h, waypoints = pos_of(state).copy(), yaw_of(state), []
    for st in plan_steps:
        if st["action"] == "forward":
            p = p + st["distance"] * np.array([math.cos(h), math.sin(h)])
            waypoints.append(p.copy())
        elif st["action"] == "turn":
            h = wrap(h + math.radians(st["angle_deg"]))

    fell = [False]

    def step(state, cmd):
        state.info["command"] = jp.array(cmd)
        act, _ = jit_inf(state.obs, rng)
        new = jit_step(state, act)
        if float(new.data.qpos[2]) < 0.5:        # 胴体が低い = 転倒
            fell[0] = True
        return new

    heading = yaw_of(state)
    wp_i = 0
    for st in plan_steps:
        if fell[0]:
            break
        a = st["action"]
        n0 = len(traj)
        if a == "forward":
            goal = waypoints[wp_i]; wp_i += 1
            start = pos_of(state)
            seg = goal - start
            L = np.linalg.norm(seg)
            if L < 0.05:
                continue
            fwd = seg / L
            best = np.inf
            for _ in range(int((L / 0.3 + 5) / env.dt)):
                if fell[0]:
                    break
                to_goal = goal - pos_of(state)
                dist = np.linalg.norm(to_goal)
                progress = (pos_of(state) - start) @ fwd
                best = min(best, dist)
                if dist < 0.4 or (progress >= L and dist > best + 0.3):   # 到着、または通り過ぎて遠ざかった
                    break
                err = wrap(math.atan2(to_goal[1], to_goal[0]) - yaw_of(state))
                v = max(vx * min(1.0, dist / slow_dist), 0.2)
                state = step(state, [v, 0.0, float(np.clip(kp*err, -1, 1))])
                traj.append(snap(state))
            rest = np.linalg.norm(goal - pos_of(state))
            print(f"  forward {st['distance']}m: {(len(traj)-n0)*env.dt:.1f}秒, 目標までの残り {rest:.2f}m")
        elif a == "turn":
            heading = wrap(heading + math.radians(st["angle_deg"]))
            for _ in range(int(brake_sec / env.dt)):           # 惰性を止めてから旋回する
                state = step(state, [0.0, 0.0, 0.0])
                traj.append(snap(state))
            n0 = len(traj)                                      # 旋回の計測はブレーキ後から
            p_t0 = pos_of(state)
            ax_t0 = np.array([math.cos(yaw_of(state)), math.sin(yaw_of(state))])
            back = 0.0
            for _ in range(int(10.0 / env.dt)):
                if fell[0]:
                    break
                err = wrap(heading - yaw_of(state))
                if abs(err) < 0.15:
                    break
                yaw_cmd = float(np.clip(kp*err, -1, 1))
                if abs(yaw_cmd) < 0.3:
                    yaw_cmd = 0.3 * np.sign(yaw_cmd)
                state = step(state, [0.0, 0.0, yaw_cmd])
                traj.append(snap(state))
                back = min(back, float((pos_of(state) - p_t0) @ ax_t0))
            err_deg = math.degrees(wrap(heading - yaw_of(state)))
            print(f"  turn {st['angle_deg']:+.0f}度: {(len(traj)-n0)*env.dt:.1f}秒, 向きの残り {err_deg:+.1f}度, 旋回中の最大後退 {-back:.2f}m")
        elif a == "stop":
            for _ in range(int(1.0 / env.dt)):
                state = step(state, [0.0, 0.0, 0.0])
                traj.append(snap(state))
    if fell[0]:
        print("  ⚠ 転倒を検出したため、プランを途中で中断しました")
    return traj, waypoints

model = env.mj_model
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, 480, 640)

def add_geom(scene, gtype, pos, size, rgba):
    if scene.ngeom >= scene.maxgeom:
        return
    mujoco.mjv_initGeom(scene.geoms[scene.ngeom], gtype,
        np.array(size, dtype=float), np.array(pos, dtype=float),
        np.eye(3).flatten(), np.array(rgba, dtype=np.float32))
    scene.ngeom += 1

def add_marker(scene, xy, rgba):
    x, y = xy
    add_geom(scene, mujoco.mjtGeom.mjGEOM_CYLINDER, [x, y, 0.01], [0.35, 0.01, 0], rgba)
    add_geom(scene, mujoco.mjtGeom.mjGEOM_CYLINDER, [x, y, 1.0],  [0.03, 1.0, 0],  rgba)

def render_plan(traj, waypoints, step=2):
    start = traj[0][0][:2]
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_TRACKING
    cam.trackbodyid = 1
    cam.distance, cam.azimuth, cam.elevation = 5.0, 135, -25
    frames, trail = [], []
    for s in traj[::step]:
        qpos, qvel = s
        data.qpos[:] = qpos
        data.qvel[:] = qvel
        mujoco.mj_forward(model, data)
        renderer.update_scene(data, camera=cam)
        add_marker(renderer.scene, start, [0.2, 0.4, 1.0, 1])
        for i, w in enumerate(waypoints):
            last = (i == len(waypoints) - 1)
            add_marker(renderer.scene, w, [1.0, 0.2, 0.2, 1] if last else [1.0, 0.6, 0.1, 1])
        trail.append(qpos[:2].copy())
        for p in trail[::3]:
            add_geom(renderer.scene, mujoco.mjtGeom.mjGEOM_SPHERE,
                     [p[0], p[1], 0.03], [0.04, 0, 0], [1.0, 0.9, 0.1, 1])
        frames.append(renderer.render())
    return frames

## 全プランを実行
`ONLY` にタグ名のリストを入れると、そのプランだけ実行します（空なら全部）。

In [ ]:
ONLY = []   # 例: ["demo1_L", "demo2_U"]

results = []
for path in sorted(glob.glob(PLAN_DIR + "/*.json")):
    tag = os.path.basename(path)[:-5]
    if ONLY and tag not in ONLY:
        continue
    d = json.load(open(path))
    print(f"=== {tag} ===")
    print("指示:", d["instruction"])
    print("プラン:", json.dumps(d["plan"], ensure_ascii=False))
    if not d["plan"]:
        print("プランが空のためスキップ\n")
        continue
    traj, waypoints = run_plan(d["plan"])
    sec = len(traj) * env.dt
    err = None
    if waypoints:
        xy = np.array([q[0][:2] for q in traj])
        err = float(np.linalg.norm(xy[-1] - waypoints[-1]))
    print(f"所要秒数: {sec:.1f} / 最終ゴール誤差 [m]: {err}")
    frames = render_plan(traj, waypoints)
    media.show_video(frames, fps=0.5 / env.dt)
    results.append((tag, d["instruction"], sec, err))

print("--- まとめ ---")
for r in results:
    print(r)

In [ ]:
def step_cmd(state, cmd, rng):
    state.info["command"] = jp.array(cmd)
    act, _ = jit_inf(state.obs, rng)
    return jit_step(state, act)

def calib(cmd, seconds=3.0, seed=0):
    rng = jax.random.PRNGKey(seed)
    state = jit_reset(rng)
    for _ in range(int(1.0 / env.dt)):                 # 1秒間立たせる
        state = step_cmd(state, [0.0, 0.0, 0.0], rng)
    y_prev, p0, total = yaw_of(state), pos_of(state), 0.0
    for _ in range(int(seconds / env.dt)):
        state = step_cmd(state, cmd, rng)
        y = yaw_of(state); total += wrap(y - y_prev); y_prev = y
    height = float(np.array(state.data.qpos)[2])
    speed = np.linalg.norm(pos_of(state) - p0) / seconds
    return total / seconds, speed, height

print(env_cfg.command_config if hasattr(env_cfg, "command_config") else "command_config なし")
print(f"{'指令 [vx, vy, yaw]':<22}{'実yaw[rad/s]':>14}{'実速度[m/s]':>14}{'胴体高さ[m]':>14}")
for cmd in ([0,0,0.5], [0,0,1.0], [0.3,0,0.5], [0.5,0,0.5], [0.5,0,1.0], [0.8,0,0.0]):
    yr, sp, h = calib(cmd)
    print(f"{str(cmd):<22}{yr:>14.2f}{sp:>14.2f}{h:>14.2f}")

a:
- 1.0
- 0.8
- 1.0
b:
- 0.9
- 0.25
- 0.5

指令 [vx, vy, yaw]         実yaw[rad/s]      実速度[m/s]       胴体高さ[m]
Module _next_time_builder__locals___next_time_bc454e15 bc454e1 load on device 'cuda:0' took 173.18 ms  (compiled)
[0, 0, 0.5]                     0.03          0.05          0.76
[0, 0, 1.0]                     0.09          0.05          0.76
[0.3, 0, 0.5]                   0.05          0.23          0.76
[0.5, 0, 0.5]                   0.08          0.37          0.77
[0.5, 0, 1.0]                   0.15          0.37          0.77
[0.8, 0, 0.0]                   0.04          0.56          0.77


### 新しいポリシーで再検証

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
BASE = "/content/drive/MyDrive/physical-ai-final"
PARAMS_PATH = BASE + "/ckpt_g1/params_v2.pkl"   #V2のポリシーを使用
PLAN_DIR = BASE + "/plans"
ENV_NAME = "G1JoystickFlatTerrain"

from google.colab import drive
drive.mount('/content/drive')
print("params:", os.path.exists(PARAMS_PATH))
print("plans:", sorted(os.listdir(PLAN_DIR)) if os.path.exists(PLAN_DIR) else "なし")

!pip install -q mujoco mujoco_mjx brax playground mediapy
!pip install -q -U flax

Mounted at /content/drive
params: True
plans: ['.ipynb_checkpoints', 'demo1_L.json', 'demo2_U.json', 'demo3_square.json', 'demo4_zigzag.json', 'demo5_short.json']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.6/30.6 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.9/356.9 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import math, json, pickle, glob
import numpy as np, jax, jax.numpy as jp
import mujoco, mediapy as media
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics
from mujoco_playground import registry
from mujoco_playground.config import locomotion_params

print(jax.devices())
env_cfg = registry.get_default_config(ENV_NAME)
env = registry.load(ENV_NAME, config=env_cfg)

ppo_params = locomotion_params.brax_ppo_config(ENV_NAME)
nf = dict(ppo_params.network_factory) if "network_factory" in ppo_params else {}
state0 = jax.jit(env.reset)(jax.random.PRNGKey(0))
obs_shape = jax.tree_util.tree_map(lambda x: x.shape, state0.obs)
network = ppo_networks.make_ppo_networks(
    obs_shape, env.action_size,
    preprocess_observations_fn=running_statistics.normalize, **nf)
make_inference_fn = ppo_networks.make_inference_fn(network)

params = pickle.load(open(PARAMS_PATH, "rb"))
jit_inf   = jax.jit(make_inference_fn(params, deterministic=True))
jit_reset = jax.jit(env.reset)
jit_step  = jax.jit(env.step)
print("policy loaded")

[CudaDevice(id=0)]
mujoco_menagerie not found. Downloading...


Cloning mujoco_menagerie: ██████████| 100/100 [00:37<00:00]


Checking out commit 1b86ece576591213e2b666ebf59508454200ca97
Successfully downloaded mujoco_menagerie
Warp 1.18.0 initialized:
   CUDA Toolkit 13.4, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.18.0
Module mujoco.mjx.warp.ffi 4443472 load on device 'cuda:0' took 698.91 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 43a6a77 load on device 'cuda:0' took 7867.86 ms  (compiled)
Module _nxn_broadphase__locals__kernel_95773766 9577376 load on device 'cuda:0' took 544.39 ms  (compiled)
Module ccd_kernel_builder__locals__ccd_kernel_1a0e5f50 1a0e5f5 load on device 'cuda:0' took 29140.15 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_dfed0db2 dfed0db load on device 'cuda:0' took 1793.82 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 8d4d513 load on device 'cuda:0' took 111.18 ms  (compiled)
Module _friction_dof_

In [ ]:
def yaw_of(s):
    w, x, y, z = np.array(s.data.qpos)[3:7]
    return math.atan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))

def pos_of(s):
    return np.array(s.data.qpos)[:2]

def wrap(a):
    return (a + math.pi) % (2*math.pi) - math.pi

def snap(s):   # GPU上のstateは保持せず、描画に必要な最小限だけCPUへ
    return np.array(s.data.qpos), np.array(s.data.qvel)

def run_plan(plan_steps, vx=0.8, kp=2.0, seed=0, brake_sec=0.0, slow_dist=1.5):
    rng = jax.random.PRNGKey(seed)
    state = jit_reset(rng)
    traj = [snap(state)]

    # 目標は最初に「理想の経路」として固定する（実際の位置のずれに引きずられない）
    p, h, waypoints = pos_of(state).copy(), yaw_of(state), []
    for st in plan_steps:
        if st["action"] == "forward":
            p = p + st["distance"] * np.array([math.cos(h), math.sin(h)])
            waypoints.append(p.copy())
        elif st["action"] == "turn":
            h = wrap(h + math.radians(st["angle_deg"]))

    fell = [False]

    def step(state, cmd):
        state.info["command"] = jp.array(cmd)
        act, _ = jit_inf(state.obs, rng)
        new = jit_step(state, act)
        if float(new.data.qpos[2]) < 0.5:        # 胴体が低い = 転倒
            fell[0] = True
        return new

    heading = yaw_of(state)
    wp_i = 0
    for st in plan_steps:
        if fell[0]:
            break
        a = st["action"]
        n0 = len(traj)
        if a == "forward":
            goal = waypoints[wp_i]; wp_i += 1
            start = pos_of(state)
            seg = goal - start
            L = np.linalg.norm(seg)
            if L < 0.05:
                continue
            fwd = seg / L
            best = np.inf
            for _ in range(int((L / 0.3 + 5) / env.dt)):
                if fell[0]:
                    break
                to_goal = goal - pos_of(state)
                dist = np.linalg.norm(to_goal)
                progress = (pos_of(state) - start) @ fwd
                best = min(best, dist)
                if dist < 0.4 or (progress >= L and dist > best + 0.3):   # 到着、または通り過ぎて遠ざかった
                    break
                err = wrap(math.atan2(to_goal[1], to_goal[0]) - yaw_of(state))
                v = max(vx * min(1.0, dist / slow_dist), 0.2)
                state = step(state, [v, 0.0, float(np.clip(kp*err, -1, 1))])
                traj.append(snap(state))
            rest = np.linalg.norm(goal - pos_of(state))
            print(f"  forward {st['distance']}m: {(len(traj)-n0)*env.dt:.1f}秒, 目標までの残り {rest:.2f}m")
        elif a == "turn":
            heading = wrap(heading + math.radians(st["angle_deg"]))
            for _ in range(int(brake_sec / env.dt)):           # 惰性を止めてから旋回する
                state = step(state, [0.0, 0.0, 0.0])
                traj.append(snap(state))
            n0 = len(traj)                                      # 旋回の計測はブレーキ後から
            p_t0 = pos_of(state)
            ax_t0 = np.array([math.cos(yaw_of(state)), math.sin(yaw_of(state))])
            back = 0.0
            for _ in range(int(10.0 / env.dt)):
                if fell[0]:
                    break
                err = wrap(heading - yaw_of(state))
                if abs(err) < 0.15:
                    break
                yaw_cmd = float(np.clip(kp*err, -1, 1))
                if abs(yaw_cmd) < 0.3:
                    yaw_cmd = 0.3 * np.sign(yaw_cmd)
                state = step(state, [0.0, 0.0, yaw_cmd])
                traj.append(snap(state))
                back = min(back, float((pos_of(state) - p_t0) @ ax_t0))
            err_deg = math.degrees(wrap(heading - yaw_of(state)))
            print(f"  turn {st['angle_deg']:+.0f}度: {(len(traj)-n0)*env.dt:.1f}秒, 向きの残り {err_deg:+.1f}度, 旋回中の最大後退 {-back:.2f}m")
        elif a == "stop":
            for _ in range(int(1.0 / env.dt)):
                state = step(state, [0.0, 0.0, 0.0])
                traj.append(snap(state))
    if fell[0]:
        print("  ⚠ 転倒を検出したため、プランを途中で中断しました")
    return traj, waypoints

model = env.mj_model
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, 480, 640)

def add_geom(scene, gtype, pos, size, rgba):
    if scene.ngeom >= scene.maxgeom:
        return
    mujoco.mjv_initGeom(scene.geoms[scene.ngeom], gtype,
        np.array(size, dtype=float), np.array(pos, dtype=float),
        np.eye(3).flatten(), np.array(rgba, dtype=np.float32))
    scene.ngeom += 1

def add_marker(scene, xy, rgba):
    x, y = xy
    add_geom(scene, mujoco.mjtGeom.mjGEOM_CYLINDER, [x, y, 0.01], [0.35, 0.01, 0], rgba)
    add_geom(scene, mujoco.mjtGeom.mjGEOM_CYLINDER, [x, y, 1.0],  [0.03, 1.0, 0],  rgba)

def render_plan(traj, waypoints, step=2):
    start = traj[0][0][:2]
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_TRACKING
    cam.trackbodyid = 1
    cam.distance, cam.azimuth, cam.elevation = 5.0, 135, -25
    frames, trail = [], []
    for s in traj[::step]:
        qpos, qvel = s
        data.qpos[:] = qpos
        data.qvel[:] = qvel
        mujoco.mj_forward(model, data)
        renderer.update_scene(data, camera=cam)
        add_marker(renderer.scene, start, [0.2, 0.4, 1.0, 1])
        for i, w in enumerate(waypoints):
            last = (i == len(waypoints) - 1)
            add_marker(renderer.scene, w, [1.0, 0.2, 0.2, 1] if last else [1.0, 0.6, 0.1, 1])
        trail.append(qpos[:2].copy())
        for p in trail[::3]:
            add_geom(renderer.scene, mujoco.mjtGeom.mjGEOM_SPHERE,
                     [p[0], p[1], 0.03], [0.04, 0, 0], [1.0, 0.9, 0.1, 1])
        frames.append(renderer.render())
    return frames

In [ ]:
ONLY = []   # 例: ["demo1_L", "demo2_U"]

results = []
for path in sorted(glob.glob(PLAN_DIR + "/*.json")):
    tag = os.path.basename(path)[:-5]
    if ONLY and tag not in ONLY:
        continue
    d = json.load(open(path))
    print(f"=== {tag} ===")
    print("指示:", d["instruction"])
    print("プラン:", json.dumps(d["plan"], ensure_ascii=False))
    if not d["plan"]:
        print("プランが空のためスキップ\n")
        continue
    traj, waypoints = run_plan(d["plan"])
    sec = len(traj) * env.dt
    err = None
    if waypoints:
        xy = np.array([q[0][:2] for q in traj])
        err = float(np.linalg.norm(xy[-1] - waypoints[-1]))
    print(f"所要秒数: {sec:.1f} / 最終ゴール誤差 [m]: {err}")
    frames = render_plan(traj, waypoints)
    media.show_video(frames, fps=0.5 / env.dt)
    results.append((tag, d["instruction"], sec, err))

print("--- まとめ ---")
for r in results:
    print(r)

In [ ]:
def step_cmd(state, cmd, rng):
    state.info["command"] = jp.array(cmd)
    act, _ = jit_inf(state.obs, rng)
    return jit_step(state, act)

def calib(cmd, seconds=3.0, seed=0):
    rng = jax.random.PRNGKey(seed)
    state = jit_reset(rng)
    for _ in range(int(1.0 / env.dt)):                 # 1秒間立たせる
        state = step_cmd(state, [0.0, 0.0, 0.0], rng)
    y_prev, p0, total = yaw_of(state), pos_of(state), 0.0
    for _ in range(int(seconds / env.dt)):
        state = step_cmd(state, cmd, rng)
        y = yaw_of(state); total += wrap(y - y_prev); y_prev = y
    height = float(np.array(state.data.qpos)[2])
    speed = np.linalg.norm(pos_of(state) - p0) / seconds
    return total / seconds, speed, height

print(env_cfg.command_config if hasattr(env_cfg, "command_config") else "command_config なし")
print(f"{'指令 [vx, vy, yaw]':<22}{'実yaw[rad/s]':>14}{'実速度[m/s]':>14}{'胴体高さ[m]':>14}")
for cmd in ([0,0,0.5], [0,0,1.0], [0.3,0,0.5], [0.5,0,0.5], [0.5,0,1.0], [0.8,0,0.0]):
    yr, sp, h = calib(cmd)
    print(f"{str(cmd):<22}{yr:>14.2f}{sp:>14.2f}{h:>14.2f}")